# RAG - Pipeline with Offline and Online Processing

## Init

In [1]:
import os, sys
sys.path.append(os.path.join(".."))

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from src import VectorStore, FlatIndex, HNSWIndex, SentenceTransformerEmbedding, Retriever, BM25Retriever, RankFusion

C:\Users\PC\AppData\Local\Temp\ipykernel_23464\2752498169.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\code\github\ai\having_fun\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\code\github\ai\rag\notebooks\..\src\rank_fusion.py:2: SyntaxWarning: invalid escape sequence '\s'
  """


## Configs

In [2]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = SentenceTransformerEmbedding
INDEX_TYPE = FlatIndex
OUTPUT_DIR = "vector_db"

## Pipeline

### Offline Processing

#### Load PDF

In [3]:
def load_document(file_path: str):
    """
    USAGE: documents = load_document(file_path)
    """
    loader = PyPDFLoader(file_path)
    return loader.load()

#### Verify

In [4]:
def verify(results):
    for i, result in enumerate(results):
        print(f"Rank {i+1}")
        print("Score: ", result.score)
        print("Metadata: ", result.document.metadata)
        print(result.document.page_content[:250])
        print("="*10)

#### Full Offline Process

In [4]:
def process_document(pdf_path: str, output_dir=OUTPUT_DIR, embedding_model=EMBEDDING_MODEL, index=INDEX_TYPE, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    documents = load_document(pdf_path)
    store = VectorStore(embedding_model=embedding_model(), index=index())
    store.index_documents(documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    store.save(output_dir)
    
    return store

In [7]:
def build_bm25(pdf_path: str):
    documents = load_document(pdf_path)
    bm25 = BM25Retriever()
    bm25.build(documents)
    return bm25

### Online Processing

#### Prompt Generation

In [6]:
def generate_prompt(docs, query: str):
    prompt = '''Given this context: \n'''
    for doc in docs:
        prompt += "#"*10 + "\n"
        prompt += "Title: " + doc.chunk.metadata["title"] + " - Page " + str(doc.chunk.metadata["page"]) + "\n"
        prompt += doc.chunk.text + "\n"
    
    prompt += "#"*10 + "\n"
    prompt += "Question:" + "\n"
    prompt += query
    return prompt


## Test

In [20]:
pdf_path = r"G:\My Drive\0-Studies\algorithm\clrs\Introduction.to.Algorithms.4th.Edition.pdf"

In [21]:
store = process_document(pdf_path=pdf_path)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3534.05it/s]


In [22]:
# Load vector db
store = VectorStore(embedding_model=EMBEDDING_MODEL, index=INDEX_TYPE).load("vector_db", embedding_model=EMBEDDING_MODEL(), index_class=INDEX_TYPE)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2952.77it/s]


In [10]:
bm25 = build_bm25(pdf_path)

AttributeError: 'Document' object has no attribute 'text'

In [4]:
query = "Dijkstra's algorithm?"

In [16]:
# Search

results = store.search(
    query="Lagrange",
    k=5
)

print(results)

[SearchResult(chunk=Chunk(id=1328, text='see (12.26c). The Lagrangian is then given by\nL(w, b, ξ, α, γ) = 1\n2 ∥w∥2 + C\nNX\nn=1\nξn (12.34)\n−\nNX\nn=1\nαn(yn(⟨w, xn⟩ + b) − 1 + ξn)\n| {z }\nconstraint (12.26b)\n−\nNX\nn=1\nγnξn\n| {z }\nconstraint (12.26c)\n.\n©2024 M. P. Deisenroth, A. A. Faisal, C. S. Ong. Published by Cambridge University Press (2020).', metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-01-15T15:41:49+00:00', 'author': 'Marc Peter Deisenroth, A. Aldo Faisal, Cheng Soon Ong', 'keywords': '', 'moddate': '2024-01-15T15:41:49+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.23 (TeX Live 2021) kpathsea version 6.3.3', 'subject': '', 'title': 'Mathematics for Machine Learning', 'trapped': '/False', 'source': 'G:\\My Drive\\0-Studies\\ml\\books\\mml\\mml-book.pdf', 'total_pages': 417, 'page': 388, 'page_label': '383'}), score=np.float32(0.4142998)), SearchResult(chunk=Chunk(id=805, text='Lagrangian\nD

In [19]:
print(len(results))

0


In [18]:
prompt = generate_prompt(results, "Dijkstra's algorithm?")
print(prompt)

Given this context: 
##########
Title: Introduction to Algorithms, Fourth Edition - Page 807
Just as in Prim’s algorithm, the running time of Dijkstra’s algorithm
depends on the speciﬁc implementation of the min-priority queue Q. A
simple implementation takes advantage of the vertices being numbered
##########
Title: Introduction to Algorithms, Fourth Edition - Page 805
Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The
shortest-path estimates appear within the vertices, and blue edges indicate predecessor values.
Blue vertices belong to the set S, and tan vertices are in the min-priority queue Q = V − S. (a)
The situation just before the ﬁrst iteration of the while loop of lines 6–12. (b)–(f) The situation
after each successive iteration of the while loop. In each part, the vertex highlighted in orange
was chosen as vertex u in line 7, and each edge highlighted in orange caused a d value and a
predecessor to change when the edge was relaxed. Th

In [23]:
# Using Retriever

query = "Dijkstra's algorithm"
retriever = Retriever(store)
results = retriever.retrieve(query, k=5)
print(results)

[SearchResult(chunk=Chunk(id=2035, text='Just as in Prim’s algorithm, the running time of Dijkstra’s algorithm\ndepends on the speciﬁc implementation of the min-priority queue Q. A\nsimple implementation takes advantage of the vertices being numbered', metadata={'producer': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creator': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creationdate': '2022-04-10T16:55:05+00:00', 'author': 'Thomas H. Cormen;Charles E. Leiserson;Ronald L. Rivest;Clifford Stein; & Charles E. Leiserson & Ronald L. Rivest & Clifford Stein', 'moddate': '2022-04-10T12:55:06-04:00', 'title': 'Introduction to Algorithms, Fourth Edition', 'source': 'G:\\My Drive\\0-Studies\\algorithm\\clrs\\Introduction.to.Algorithms.4th.Edition.pdf', 'total_pages': 1677, 'page': 807, 'page_label': '808'}), score=np.float32(0.7165558)), SearchResult(chunk=Chunk(id=2028, text='Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The\nshortest-path estim

In [8]:
prompt = generate_prompt(results, query)
print(prompt)

Given this context: 
##########
Title: Introduction to Algorithms, Fourth Edition - Page 805
Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The
shortest-path estimates appear within the vertices, and blue edges indicate predecessor values.
Blue vertices belong to the set S, and tan vertices are in the min-priority queue Q = V − S. (a)
The situation just before the ﬁrst iteration of the while loop of lines 6–12. (b)–(f) The situation
after each successive iteration of the while loop. In each part, the vertex highlighted in orange
was chosen as vertex u in line 7, and each edge highlighted in orange caused a d value and a
predecessor to change when the edge was relaxed. The d values and predecessors shown in part
(f) are the ﬁnal values.
Because Dijkstra’s algorithm always chooses the “lightest” or
##########
Title: Introduction to Algorithms, Fourth Edition - Page 835
size of the input, it can be reduced to be linear in the size of the input
usin

## Load and Query

In [25]:
db_path = r"G:\My Drive\0-Studies\vector_db\maths_for_ml"

store = VectorStore(embedding_model=EMBEDDING_MODEL, index=INDEX_TYPE).load(db_path, embedding_model=EMBEDDING_MODEL(), index_class=INDEX_TYPE)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4009.55it/s]


In [28]:
query = "Lagrange Multipliers"
results = store.search(query, k=5)

print(len(results))

5


In [30]:
for result in results:
    print(f"Score: ", result.score)

Score:  0.51307493
Score:  0.4998095
Score:  0.48970145
Score:  0.48765373
Score:  0.48701757


In [29]:
print(results)

[SearchResult(chunk=Chunk(id=799, text='Consider the special case when all the preceding functions are linear, i.e.,\nmin\nx∈Rd\nc⊤x (7.39)\nsubject to Ax ⩽ b ,\nwhere A ∈ Rm×d and b ∈ Rm. This is known as a linear program. It has d linear program\nLinear programs are\none of the most\nwidely used\napproaches in\nindustry .\nvariables and m linear constraints. The Lagrangian is given by\nL(x, λ) = c⊤x + λ⊤(Ax − b) , (7.40)\nwhere λ ∈ Rm is the vector of non-negative Lagrange multipliers. Rear-\nranging the terms corresponding to x yields\nL(x, λ) = (c + A⊤λ)⊤x − λ⊤b . (7.41)\nTaking the derivative of L(x, λ) with respect to x and setting it to zero\ngives us\nc + A⊤λ = 0 . (7.42)\nTherefore, the dual Lagrangian is D(λ) = −λ⊤b. Recall we would like\nto maximize D(λ). In addition to the constraint due to the derivative of', metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-01-15T15:41:49+00:00', 'author': 'Marc Peter Deisenroth, A. Aldo Faisa

In [34]:
query = "Lagrange Multipliers"

retriever = Retriever(store, score_threshold=.4)

results = retriever.retrieve(query, k=5)

print(len(results))

5


In [33]:
results = store.search(query, k=10)

candidates = [
    r for r in results if r.score >= 0.4
]

candidates[:5]

[SearchResult(chunk=Chunk(id=799, text='Consider the special case when all the preceding functions are linear, i.e.,\nmin\nx∈Rd\nc⊤x (7.39)\nsubject to Ax ⩽ b ,\nwhere A ∈ Rm×d and b ∈ Rm. This is known as a linear program. It has d linear program\nLinear programs are\none of the most\nwidely used\napproaches in\nindustry .\nvariables and m linear constraints. The Lagrangian is given by\nL(x, λ) = c⊤x + λ⊤(Ax − b) , (7.40)\nwhere λ ∈ Rm is the vector of non-negative Lagrange multipliers. Rear-\nranging the terms corresponding to x yields\nL(x, λ) = (c + A⊤λ)⊤x − λ⊤b . (7.41)\nTaking the derivative of L(x, λ) with respect to x and setting it to zero\ngives us\nc + A⊤λ = 0 . (7.42)\nTherefore, the dual Lagrangian is D(λ) = −λ⊤b. Recall we would like\nto maximize D(λ). In addition to the constraint due to the derivative of', metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-01-15T15:41:49+00:00', 'author': 'Marc Peter Deisenroth, A. Aldo Faisa

In [35]:
chunks = [result.chunk for result in results]
print(len(chunks))

5


In [37]:
for chunk in chunks:
    print(chunk.text)

Consider the special case when all the preceding functions are linear, i.e.,
min
x∈Rd
c⊤x (7.39)
subject to Ax ⩽ b ,
where A ∈ Rm×d and b ∈ Rm. This is known as a linear program. It has d linear program
Linear programs are
one of the most
widely used
approaches in
industry .
variables and m linear constraints. The Lagrangian is given by
L(x, λ) = c⊤x + λ⊤(Ax − b) , (7.40)
where λ ∈ Rm is the vector of non-negative Lagrange multipliers. Rear-
ranging the terms corresponding to x yields
L(x, λ) = (c + A⊤λ)⊤x − λ⊤b . (7.41)
Taking the derivative of L(x, λ) with respect to x and setting it to zero
gives us
c + A⊤λ = 0 . (7.42)
Therefore, the dual Lagrangian is D(λ) = −λ⊤b. Recall we would like
to maximize D(λ). In addition to the constraint due to the derivative of
7.2 Constrained Optimization and Lagrange Multipliers 233
Figure 7.4
Illustration of
constrained
optimization. The
unconstrained
problem (indicated
by the contour
lines) has a
minimum on the
right side (indicated
by the circle).